<a href="https://colab.research.google.com/github/Jeremy-Gitau/on-device-ml-kotlin/blob/main/notebooks/01_mobilenet_v2_to_tflite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LiteRT Part 2: MobileNet V2 to LiteRT
Converting a pretrained TensorFlow 2 MobileNet V2 classifier into a `.tflite` file for Android.

# Import libraries

First we import the necessary libraries to load the model and to convert the model to .tflite
* Kagglehub is Kaggle's library for fetching models and datasets.
* TensorFlow is the ML framework the model was built with.


In [1]:
import kagglehub
import tensorflow as tf

# Now we load the model.


* MODEL_HANDLE stores the model's address on Kaggle ("google/mobilenet-v2/tensorFlow2/100-224-classification").
Let's break down the variation:
    * 100 means width multiplier 1.00, the full-size model. A value of 100 means every layer uses 100% of its standard neural width (channels) to look for complex visual patterns.
    * 224 means it expects 224 × 224 pixel images.
    * Classification means it outputs "what is this?" rather than drawing boxes like a detection model.
* IMG_SIZE is a constant, so the number 224 lives in one place instead of being repeated everywhere.


In [2]:
MODEL_HANDLE = "google/mobilenet-v2/tensorFlow2/100-224-classification"
IMG_SIZE = 224

The line `model_path = kagglehub.model_download(MODEL_HANDLE)` automatically finds, downloads, and saves the MobileNet files from Kaggle's servers.

The line `tf.saved_model.load` loads the raw, compiled computational graph.

To load a flexible model ready for training or fine-tuning, you would use `tf.keras.models.load_model`, but we are not training or finetuning. The goal is to load the model and convert to a LiteRT model for Android to consume it.

In [3]:
model_path = kagglehub.model_download(MODEL_HANDLE)
model = tf.saved_model.load(model_path)
print("Loaded from:", model_path)


  0%|          | 0.00/13.5M [00:00<?, ?B/s]

  0%|          | 0.00/1.25M [00:00<?, ?B/s]




100%|██████████| 5.18k/5.18k [00:00<00:00, 5.95MB/s]


 80%|███████▉  | 1.00M/1.25M [00:00<00:00, 2.45MB/s]
100%|██████████| 1.25M/1.25M [00:00<00:00, 2.92MB/s]

 22%|██▏       | 3.00M/13.5M [00:00<00:01, 6.28MB/s]
 52%|█████▏    | 7.00M/13.5M [00:00<00:00, 14.6MB/s]
100%|██████████| 13.5M/13.5M [00:00<00:00, 17.4MB/s]


Loaded from: /root/.cache/kagglehub/models/google/mobilenet-v2/tensorFlow2/100-224-classification/2


## Convert to LiteRT (.tflite)

Conversion translates the TensorFlow model into the `.tflite` format, which the LiteRT runtime can run on phones, tablets and embedded devices.

Conversion on its own doesn't make the model much smaller, so expect a file of roughly the same size (about 14 MB). Shrinking it and speeding it up is done with **quantization**, which we'll cover later in the series.

### Describing the input

The converter needs a fixed, known graph to work with, so we wrap the model in a small function and describe exactly what it accepts.

* **`@tf.function`** turns the Python function into a TensorFlow graph, a fixed sequence of operations the converter can read and translate.
* **`input_signature`** locks in the exact shape and type of input the model will receive. Phone runtimes work best when memory sizes are known ahead of time.
* **`[1, IMG_SIZE, IMG_SIZE, 3]`** is the input shape:
  * `1`: batch size, one image at a time
  * `IMG_SIZE, IMG_SIZE`: height and width, 224 × 224 pixels
  * `3`: colour channels, red, green and blue
* **`tf.float32`** means each pixel value is a decimal number (like 0.25 or 1.0), not a whole number from 0 to 255.

### Converting and saving

* **`serve.get_concrete_function()`** builds the graph for that one input signature.
* **`TFLiteConverter.from_concrete_functions(...)`** creates the converter. Passing `model` keeps the trained weights attached so they're packed into the file.
* **`converter.convert()`** does the translation and returns the model as bytes.
* **`open(..., "wb")`** writes those bytes to `mobilenet_v2.tflite` in binary mode.

This `[1, 224, 224, 3]` float32 input is the same shape our Android app will need to provide.

In [4]:
@tf.function(input_signature=[tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32)])
def serve(images):
    return model(images)

converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [serve.get_concrete_function()], model
)
tflite_model = converter.convert()

with open("mobilenet_v2.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Saved mobilenet_v2.tflite ({len(tflite_model) / 1e6:.1f} MB)")

Saved mobilenet_v2.tflite (14.0 MB)


# Load Model and Allocate Memory

Because standard TensorFlow models are too heavy for mobile devices, LiteRT doesn't use the standard heavy runtime engine. Instead, it uses an ultra-lean runtime engine called an Interpreter.
We use the LiteRT interpreter from the `ai_edge_litert` package because `tf.lite.Interpreter` is deprecated in favour of it.
So we initialize the engine (Interpreter) with the "`mobilenet_v2.tflite`" in the path.

## Memory allocation

On server computers, memory shifts dynamically. On mobile devices or microcontrollers, memory must be perfectly managed.
This line, `interpreter.allocate_tensors()`, tells the computer to pre-calculate and permanently reserve the exact amount of RAM needed to pass an image through all the layers of the model. This prevents your mobile app from crashing due to unexpected "out-of-memory" spikes while running a live prediction.

In [5]:
!pip install ai_edge_litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 79.2 MB/s eta 0:00:00


In [6]:
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path="mobilenet_v2.tflite")
interpreter.allocate_tensors()

for t in interpreter.get_input_details():
    print("Input: ", t["shape"], t["dtype"].__name__)
for t in interpreter.get_output_details():
    print("Output:", t["shape"], t["dtype"].__name__)

Input:  [  1 224 224   3] float32
Output: [   1 1001] float32


# Download Labels

Now we download a text file containing the 1,01 category labels used by ImageNet models (like MobileNet) and load them into a Python list.

In [7]:
import urllib.request

urllib.request.urlretrieve(
    "https://storage.googleapis.com/download.tensorflow.org/data/ImageNetLabels.txt",
    "labels.txt",
)
labels = open("labels.txt").read().splitlines()
print(len(labels), "labels, first few:", labels[:3])

1001 labels, first few: ['background', 'tench', 'goldfish']


# Run Inference on the Test Image

We finalize the pipeline by downloading a test image for Grace Hopper, the computer science pioneer, wearing her US Navy uniform, preprocessing it to match our network's constraints, and running it through our optimized LiteRT Interpreter.
* Preprocessing: The image is resized to (224, 224), normalized to a 0.0 - 1.0 pixel range, and expanded to a 4D batch shape (1, 224, 224, 3).
* Execution: We load the image into the interpreter's input tensor slot and trigger the forward pass via .invoke().
* Output: The model returns raw numerical scores (logits). We pass these through a softmax function to convert them into a distribution of percentages that sum to 100%.
* Prediction: The script sorts the probabilities in descending order and prints out the top 3 most confident categories cross-referenced against our downloaded text labels.

In [8]:
import numpy as np

img_path = tf.keras.utils.get_file(
    "grace_hopper.jpg",
    "https://storage.googleapis.com/download.tensorflow.org/example_images/grace_hopper.jpg",
)
img = tf.keras.utils.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
x = np.expand_dims(np.array(img, dtype=np.float32) / 255.0, 0)  # pixels in 0 to 1

interpreter.set_tensor(interpreter.get_input_details()[0]["index"], x)
interpreter.invoke()
logits = interpreter.get_tensor(interpreter.get_output_details()[0]["index"])[0]
probs = tf.nn.softmax(logits).numpy()

for i in probs.argsort()[-3:][::-1]:
    print(f"{labels[i]:<20} {probs[i] * 100:.1f}%")

61306/61306 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
military uniform     88.2%
suit                 1.3%
Windsor tie          0.6%


## Try It Yourself

Run the next two cells in Colab to open a small demo where you can paste any image URL and see the model's top predictions. The interactive demo only works when the notebook is running, so it won't appear if you're viewing this on GitHub.

---

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf
import urllib.request
from PIL import Image
import io

def classify_image_by_url(image_url):
    if not image_url or not image_url.strip():
        return None, "Error: Please paste an image URL!"

    try:
        # Fetch image from URL
        req = urllib.request.Request(image_url.strip(), headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            image_bytes = response.read()
        img_source = Image.open(io.BytesIO(image_bytes))
    except Exception as e:
        return None, f"Error fetching image from URL: {str(e)}"

    try:
        # Convert to numpy array and preprocess
        img_arr = np.array(img_source, dtype=np.float32)
        img_resized = tf.image.resize(img_arr, (IMG_SIZE, IMG_SIZE))
        x = np.expand_dims(img_resized / 255.0, axis=0)

        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()

        interpreter.set_tensor(input_details[0][0], x) if isinstance(input_details[0], tuple) else interpreter.set_tensor(input_details[0]["index"], x)
        interpreter.invoke()

        logits = interpreter.get_tensor(output_details[0]["index"])[0]
        probs = tf.nn.softmax(logits).numpy()

        results = {}
        for i in probs.argsort()[-3:][::-1]:
            results[labels[i]] = float(probs[i])

        return img_source, results

    except Exception as e:
        return None, f"Error processing image: {str(e)}. Please try another URL!"

with gr.Blocks() as demo:
    gr.Markdown("# MobileNet V2 Image Classifier (LiteRT)")
    gr.Markdown("Paste an image URL to download, visualize, and classify.")

    with gr.Row():
        with gr.Column():
            url_input = gr.Textbox(
                label="Paste Image URL",
                placeholder="https://example.com/image.jpg",
                value="https://storage.googleapis.com/download.tensorflow.org/example_images/grace_hopper.jpg"
            )
            with gr.Row():
                submit_btn = gr.Button("Classify Image", variant="primary")
                clear_btn = gr.ClearButton(components=[url_input], value="Clear Input")

            image_preview = gr.Image(label="Downloaded Image Preview", type="pil")

        with gr.Column():
            output_lbl = gr.Label(num_top_classes=3, label="Top Predictions")

    submit_btn.click(fn=classify_image_by_url, inputs=url_input, outputs=[image_preview, output_lbl])

    # Clear both the preview and the labels when clicking Clear
    clear_btn.add([image_preview, output_lbl])

demo.launch()